### Voting

In [11]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import BernoulliNB,GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import VotingClassifier

In [13]:
kyphosis = pd.read_csv(r"../../Datasets/Kyphosis.csv")
kyphosis

,Kyphosis,Age,Number,Start
0,absent,71,3,5
1,absent,158,3,14
2,present,128,4,5
3,absent,2,5,1
4,absent,1,4,15
...,...,...,...,...
76,present,157,3,13
77,absent,26,7,13
78,absent,120,2,13
79,present,42,7,6


In [15]:
X = kyphosis.drop('Kyphosis', axis=1)
y = kyphosis['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= y)

In [17]:
dtc=DecisionTreeClassifier(max_depth = 4,random_state=26)
knn = KNeighborsClassifier(n_neighbors=3)
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()
voting = VotingClassifier(estimators=[('TREE',dtc),('KNN',knn),('LR',lr),('LDA',lda),('NB',nb)])
voting.fit(X_train,y_train)
y_pred = voting.predict(X_test)
accuracy_score(y_test,y_pred)

0.88

In [21]:
voting = VotingClassifier(
    estimators=[('TREE',dtc),
                ('KNN',knn),
                ('LR',lr),
                ('LDA',lda),
                ('NB',nb)],voting='soft', weights = [1,2,2,4,4] 
)
voting.fit(X_train,y_train)
y_pred_prob = voting.predict_proba(X_test)
log_loss(y_test,y_pred_prob)ka

0.4007865135748995

In [26]:
estims=[dtc,knn,lr,lda,nb]
for e in estims:
    e.fit(X_train,y_train)
    y_pred_prob = e.predict_proba(X_test)
    print('log loss of ',e,'=', log_loss(y_test,y_pred_prob))

log loss of  DecisionTreeClassifier(max_depth=4, random_state=26) = 4.582748472683514
log loss of  KNeighborsClassifier(n_neighbors=3) = 6.0123482471692045
log loss of  LogisticRegression() = 0.3357708471894506
log loss of  LinearDiscriminantAnalysis() = 0.34281059610597553
log loss of  GaussianNB() = 0.39698492604441094
